# 007 Built-in Middleware

这是 LangChain 学习线的第七份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/middleware/built-in
- https://docs.langchain.com/oss/python/langchain/agents

学习目标：

1. 理解 middleware 在 LangChain agent 中解决什么问题
2. 认识常用 built-in middleware
3. 把 middleware 和本仓库 Harness 控制面对齐
4. 判断哪些 middleware 可以直接用，哪些仍要业务系统兜底
5. 为后续把 LangChain 接入 FastAPI 做设计准备

---

## 1. Middleware 是什么

如果 agent 是一个循环：

```text
model call -> tool call -> observation -> next model call
```

middleware 就是在这个循环的关键节点插入控制逻辑。

你可以把它类比成 Java Web 里的 Filter / Interceptor：

```text
Request
  -> Filter / Interceptor
  -> Controller
  -> Service
  -> Response
```

LangChain middleware 作用在 agent runtime 里；FastAPI middleware 作用在 HTTP 请求里。不要混淆。

In [ ]:
%pip install -U langchain langchain-openai python-dotenv

## 2. 看看当前版本有哪些 Middleware

先导入一些官方 built-in middleware。

In [1]:
from inspect import signature

from langchain.agents.middleware import (
    ContextEditingMiddleware,
    FilesystemFileSearchMiddleware,
    HumanInTheLoopMiddleware,
    LLMToolSelectorMiddleware,
    ModelCallLimitMiddleware,
    ModelFallbackMiddleware,
    ModelRetryMiddleware,
    PIIMiddleware,
    ShellToolMiddleware,
    SummarizationMiddleware,
    TodoListMiddleware,
    ToolCallLimitMiddleware,
    ToolRetryMiddleware,
)

middleware_classes = [
    SummarizationMiddleware,
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    ModelFallbackMiddleware,
    ModelRetryMiddleware,
    ToolRetryMiddleware,
    PIIMiddleware,
    LLMToolSelectorMiddleware,
    ContextEditingMiddleware,
    TodoListMiddleware,
    FilesystemFileSearchMiddleware,
    ShellToolMiddleware,
]

for cls in middleware_classes:
    print(cls.__name__, signature(cls))

SummarizationMiddleware (model: str | langchain_core.language_models.chat_models.BaseChatModel, *, trigger: tuple[typing.Literal['fraction'], float] | tuple[typing.Literal['tokens'], int] | tuple[typing.Literal['messages'], int] | list[tuple[typing.Literal['fraction'], float] | tuple[typing.Literal['tokens'], int] | tuple[typing.Literal['messages'], int]] | None = None, keep: tuple[typing.Literal['fraction'], float] | tuple[typing.Literal['tokens'], int] | tuple[typing.Literal['messages'], int] = ('messages', 20), token_counter: collections.abc.Callable[[collections.abc.Iterable[langchain_core.messages.base.BaseMessage | list[str] | tuple[str, str] | str | dict[str, typing.Any]]], int] = <function count_tokens_approximately at 0x7fb52cd9dd00>, summary_prompt: str = '<role>\nContext Extraction Assistant\n</role>\n\n<primary_objective>\nYour sole objective in this task is to extract the highest quality/most relevant context from the conversation history below.\n</primary_objective>\n\n<o

## 3. Middleware 和 Harness 控制面对照

| LangChain middleware | 解决的问题 | 本仓库对应概念 |
|---|---|---|
| `SummarizationMiddleware` | 上下文太长时总结 | context compact |
| `HumanInTheLoopMiddleware` | 工具执行前人工确认 | approval checkpoint |
| `ModelCallLimitMiddleware` | 限制模型调用次数 | `max_steps` / loop guard |
| `ToolCallLimitMiddleware` | 限制工具调用次数 | tool permission / policy |
| `ModelFallbackMiddleware` | 模型失败切换备用模型 | fallback model |
| `ModelRetryMiddleware` | 模型调用失败重试 | recovery fuse |
| `ToolRetryMiddleware` | 工具失败重试 | tool recovery |
| `PIIMiddleware` | 敏感信息处理 | input/output guardrail |
| `LLMToolSelectorMiddleware` | 工具太多时筛选工具 | Skill selection / tool exposure |
| `ContextEditingMiddleware` | 编辑上下文 | context governance |
| `TodoListMiddleware` | 任务清单 | visible planning |
| `FilesystemFileSearchMiddleware` | 文件搜索工具 | repo research tool |
| `ShellToolMiddleware` | shell 工具 | high-risk tool + approval |

可以看到，LangChain middleware 已经覆盖了很多 Harness Engineering 主题。

## 4. 调用限制：ModelCallLimit / ToolCallLimit

这类 middleware 的作用是防止 agent loop 失控。

在本仓库里，我们用 `max_steps` 控制 Query Loop：

```python
while run.current_loop_step <= run.max_steps:
    ...
```

LangChain 可以通过 middleware 控制模型调用次数或工具调用次数。

In [2]:
model_limit = ModelCallLimitMiddleware(run_limit=4, exit_behavior="end")
tool_limit = ToolCallLimitMiddleware(run_limit=6, exit_behavior="continue")

print(model_limit)
print(tool_limit)

## 5. 人工审批：HumanInTheLoopMiddleware

这个 middleware 和本仓库的 approval 很像。

但要注意：

```text
LangChain HITL middleware 可以中断等待人工确认。
本仓库 Harness approval 还额外维护 run、ApprovalTicket、SSE、resume_approval。
```

所以它们解决的问题相近，但落地边界不同。

In [3]:
approval_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "shell": True,
        "write_file": True,
    },
    description_prefix="工具执行需要人工审批",
)

print(approval_middleware)

## 6. 敏感信息：PIIMiddleware

PII middleware 可以处理 email、信用卡、IP、URL 等敏感信息。

常见策略：

- `block`：阻断
- `redact`：移除或替换
- `mask`：打码
- `hash`：哈希

这类能力应该放在模型调用前后，而不是指望模型自己遵守。

In [4]:
pii_input_guard = PIIMiddleware("email", strategy="redact", apply_to_input=True)
pii_output_guard = PIIMiddleware("credit_card", strategy="mask", apply_to_output=True)

print(pii_input_guard)
print(pii_output_guard)

## 7. Retry 和 Fallback

Retry 解决短暂失败：

- 网络抖动
- 临时限流
- 工具偶发异常

Fallback 解决模型不可用或模型能力不匹配：

- 主模型失败后切备用模型
- 便宜模型不行时切强模型

注意：retry 不是无限重试；fallback 也不是替代验证。

In [5]:
model_retry = ModelRetryMiddleware(max_retries=2, initial_delay=0.5, max_delay=3.0)
tool_retry = ToolRetryMiddleware(max_retries=2, initial_delay=0.5, max_delay=3.0)

print(model_retry)
print(tool_retry)

## 8. Tool 选择：LLMToolSelectorMiddleware

当工具很多时，不应该每次都把所有工具暴露给模型。

这和本仓库的两个设计有关：

- `SkillRegistry.select_skills(...)`
- `ToolRegistry.list_openai_tools()`

LangChain 的 tool selector 可以在工具很多时先筛选一批相关工具，降低上下文成本。

In [6]:
tool_selector = LLMToolSelectorMiddleware(max_tools=4, always_include=["echo"])
print(tool_selector)

## 9. 文件搜索和 Shell：高风险能力

`FilesystemFileSearchMiddleware` 和 `ShellToolMiddleware` 很像 coding agent 的能力。

但是这类工具风险高：

- 可能读到敏感文件
- shell 可能执行危险命令
- 输出可能很大
- 需要权限边界和审计

所以在本仓库语境下，不能因为 LangChain 提供了 middleware，就直接绕过 Harness approval。

In [7]:
file_search = FilesystemFileSearchMiddleware(root_path=".", use_ripgrep=True, max_file_size_mb=2)

# ShellToolMiddleware 这里只展示对象创建，不执行任何 shell 命令。
shell_tool = ShellToolMiddleware(workspace_root=".")

print(file_search)
print(shell_tool)

## 10. 如何接入 create_agent

LangChain agent 可以接收 middleware 列表：

```python
agent = create_agent(
    model=model,
    tools=tools,
    middleware=[
        ModelCallLimitMiddleware(run_limit=4),
        ToolRetryMiddleware(max_retries=2),
        HumanInTheLoopMiddleware(interrupt_on={"shell": True}),
    ],
)
```

这说明 LangChain 正在把很多 Harness 控制面做成标准组件。

## 11. 和本仓库的设计判断

如果以后把 LangChain middleware 引入本项目，建议这样分层：

| 能力 | 可以优先用 LangChain middleware 吗 | 仍需本仓库兜底吗 |
|---|---|---|
| model retry | 可以 | 需要记录 ledger |
| tool retry | 可以 | 需要权限和失败审计 |
| call limit | 可以 | 需要业务 max_steps 对齐 |
| summarization | 可以 | 需要保留关键执行证据 |
| HITL approval | 谨慎 | 需要和现有 ApprovalTicket/SSE 恢复打通 |
| shell | 谨慎 | 必须走 approval 和 allowed_paths |
| file search | 可以试点 | 需要路径边界 |
| PII | 可以 | 需要按业务合规规则配置 |

一句话：

```text
LangChain middleware 可以减少重复造轮子，但不能替代业务边界设计。
```

## 12. 本讲小结

这一讲记住四点：

1. Middleware 是 agent runtime 的控制面扩展点。
2. LangChain built-in middleware 已覆盖 context、HITL、limit、retry、fallback、PII、tool selection 等常见问题。
3. 这些能力和本仓库 Harness Engineering 主题高度重合。
4. 真正落地时，要把 middleware 和现有 ledger、approval、policy、SSE 恢复机制对齐。

下一步可以做两件事之一：

- 继续学习 LangChain middleware 自定义扩展
- 开始设计一个 `/langchain-study` 教学 endpoint，把低风险 LangChain agent 接进 FastAPI